In [5]:
import edsl
print(f"EDSL version: {edsl.__version__}")

EDSL version: 1.0.8


In [6]:
import os
os.environ["EXPECTED_PARROT_API_KEY"] = "meeeZPjO2DQ0dlc-2AClgNaNOTLiHndnc5j7ue4KFZs" #TODO delete before submit
key = os.getenv("EXPECTED_PARROT_API_KEY")
if key:
    print(f"EXPECTED_PARROT_API_KEY is set ({len(key)} characters)")
    print("Remote inference is enabled — surveys will run on Expected Parrot's servers.")
else:
    print("EXPECTED_PARROT_API_KEY not found.")
    print("Set it using one of the options above before running any surveys.")

EXPECTED_PARROT_API_KEY is set (43 characters)
Remote inference is enabled — surveys will run on Expected Parrot's servers.


In [7]:
import numpy as np
import pandas as pd
from edsl import (
    Agent, AgentList, QuestionMultipleChoice, Survey, Model, Scenario
)

## Demographic Marginal Distribution

In [8]:
N_AGENTS = 800
RNG = np.random.default_rng(seed=2026)

DEMOS = {
    "age_bracket":   {"18-29": 0.20, "30-49": 0.33, "50-64": 0.24, "65+": 0.23},
    "gender":        {"Woman": 0.515, "Man": 0.485},
    "race":          {"White": 0.62, "Hispanic": 0.17, "Black": 0.12,
                      "Asian": 0.06, "Other": 0.03},
    "education":     {"HS or less": 0.37, "Some college": 0.28,
                      "Bachelor's": 0.21, "Postgrad": 0.14},
    "party":         {"Republican / lean R": 0.47,
                      "Democrat / lean D": 0.47,
                      "True independent": 0.06},
    "ideology":      {"Liberal": 0.25, "Moderate": 0.37,
                      "Conservative": 0.36, "Don't know": 0.02},
}

In [9]:
def sample_attr(attr):
    """Draw one categorical value from a documented marginal distribution."""
    cats = list(DEMOS[attr])
    probs = list(DEMOS[attr].values())
    return RNG.choice(cats, p=probs)

## Persona Construction


In [10]:
def build_persona():
    p = {a: sample_attr(a) for a in DEMOS}
    # Collapse education for the "college vs. no college" comparison dimension
    p["has_college_degree"] = p["education"] in ("Bachelor's", "Postgrad")
    return p

In [11]:
def persona_to_traits(p):
    college_phrase = ("a four-year college degree or more" if p["has_college_degree"]
                      else "no four-year college degree")
    party_short = p["party"].replace(" / lean ", "/lean ")
    bio = (
        f"You are a {p['age_bracket']}-year-old {p['gender'].lower()} living "
        f"in the United States. You identify as {p['race'].lower()} and you "
        f"have {college_phrase} (specifically: {p['education']}). "
        f"Politically, you describe yourself as {p['ideology'].lower()} and "
        f"you identify as {party_short}. Please answer the following public-"
        f"opinion question the way someone with this background would answer "
        f"honestly and authentically, choosing the single option that best "
        f"reflects your view."
    )
    return {
        "age_bracket":         p["age_bracket"],
        "gender":              p["gender"],
        "race_ethnicity":      p["race"],
        "education_level":     p["education"],
        "has_college_degree":  str(p["has_college_degree"]),
        "party_affiliation":   p["party"],
        "political_ideology":  p["ideology"],
        "persona":             bio,
    }

In [12]:
agents = AgentList([
    Agent(traits=persona_to_traits(build_persona()))
    for _ in range(N_AGENTS)
])
print(f"Constructed {len(agents)} agent personas (target = {N_AGENTS}).")

Constructed 800 agent personas (target = 800).


## Survey Question

In [13]:
q_conf = QuestionMultipleChoice(
    question_name="trust_scientists",
    question_text=(
        "How much confidence, if any, do you have in scientists to act in "
        "the best interests of the public?"
    ),
    question_options=[
        "A great deal of confidence",
        "A fair amount of confidence",
        "Not too much confidence",
        "No confidence at all",
    ],
)
survey = Survey(questions=[q_conf])

## Run survey

In [14]:
model = Model("gpt-4o", temperature=1.0)
results = survey.by(agents).by(model).run()

## Analysis

In [15]:
df = results.select(
    "agent.party_affiliation", "agent.race_ethnicity",
    "agent.education_level", "agent.has_college_degree",
    "answer.trust_scientists"
).to_pandas()

In [16]:
df["at_least_fair"] = df["answer.trust_scientists"].isin(
    ["A great deal of confidence", "A fair amount of confidence"]).astype(int)
df["great_deal"] = (df["answer.trust_scientists"]
                    == "A great deal of confidence").astype(int)

In [17]:
def simple_party(p):
    if "Republican" in p: return "Republicans"
    if "Democrat" in p:   return "Democrats"
    return "Independents"
df["party_simple"] = df["agent.party_affiliation"].apply(simple_party)

In [18]:
#overall 4-cat distribution
print(df["answer.trust_scientists"].value_counts(normalize=True).mul(100).round(1))

answer.trust_scientists
A fair amount of confidence    75.5
Not too much confidence        17.1
A great deal of confidence      7.4
Name: proportion, dtype: float64


In [19]:
#d1:party
print(df.groupby("party_simple")["at_least_fair"].mean().mul(100).round(1))

party_simple
Democrats       100.0
Independents     67.4
Republicans      66.6
Name: at_least_fair, dtype: float64


In [20]:
#d2 race
print(df.groupby("agent.race_ethnicity")["at_least_fair"].mean().mul(100).round(1))

agent.race_ethnicity
Asian       83.3
Black       82.7
Hispanic    86.8
Other       79.2
White       81.9
Name: at_least_fair, dtype: float64


In [21]:
#d3 education
print(df.groupby("agent.has_college_degree")["great_deal"].mean().mul(100).round(1))

agent.has_college_degree
False     4.8
True     12.4
Name: great_deal, dtype: float64


In [22]:
#d4 party * education
print(df.groupby(["party_simple", "agent.has_college_degree"])
        ["great_deal"].mean().mul(100).round(1))

party_simple  agent.has_college_degree
Democrats     False                        9.2
              True                        24.8
Independents  False                        3.3
              True                        12.5
Republicans   False                        0.0
              True                         0.0
Name: great_deal, dtype: float64


In [23]:
#d5 race within democrats
print(df[df["party_simple"] == "Democrats"]
        .groupby("agent.race_ethnicity")["great_deal"].mean().mul(100).round(1))

agent.race_ethnicity
Asian       16.7
Black        4.1
Hispanic     9.1
Other       10.0
White       18.3
Name: great_deal, dtype: float64


In [24]:
df.to_csv("ai_responses_raw.csv", index=False)